# Avant le TP4 : observer la réplication et piloter les nœuds depuis le notebook

Le TP4 arrête un nœud du cluster pendant qu'un producteur écrit en continu. Cette démonstration montre les outils du notebook qui servent à cela, sur un topic jetable `demo-tp4`, sans arrêter de nœud :

1. lire leader, répliques et ISR de chaque partition en Python ;
2. savoir d'où vient la valeur d'un paramètre (topic, cluster, défaut) ;
3. faire tourner un producteur en arrière-plan et suivre ses compteurs ;
4. agir sur les conteneurs du lab avec `docker compose`.

Sorties de référence : exécution du 07/11/2026 22:41 UTC sur le lab.

In [1]:
import os
import threading
import time

from confluent_kafka import Producer
from confluent_kafka.admin import AdminClient, ConfigResource, ConfigSource, NewTopic

BS = os.environ["BS"]
admin = AdminClient({"bootstrap.servers": BS})

admin.create_topics([NewTopic("demo-tp4", num_partitions=3, replication_factor=3,
                                config={"min.insync.replicas": "2"})])["demo-tp4"].result()
print("demo-tp4 cree")

demo-tp4 cree


## 1. Leader, répliques et ISR

`list_topics()` renvoie les métadonnées du cluster. Pour chaque partition : `leader`, `replicas` (le placement prévu, le premier est le leader préféré) et `isrs` (les répliques synchronisées).

In [2]:
topic = admin.list_topics("demo-tp4", timeout=10).topics["demo-tp4"]
print("partition leader repliques isr")
for p in sorted(topic.partitions.values(), key=lambda p: p.id):
    print("%-9d %-6d %-9s %s" % (p.id, p.leader, p.replicas, p.isrs))

partition leader repliques isr
0         2      [2, 3, 1] [2, 3, 1]
1         3      [3, 1, 2] [3, 1, 2]
2         1      [1, 2, 3] [1, 2, 3]


In [3]:
!docker exec kafka-tools kafka-topics.sh --bootstrap-server $BS --describe --topic demo-tp4

Topic: demo-tp4	TopicId: kMnevafiT1C0I4MsDMrMzA	PartitionCount: 3	ReplicationFactor: 3	Configs: min.insync.replicas=2,segment.bytes=1073741824,unclean.leader.election.enable=false
	Topic: demo-tp4	Partition: 0	Leader: 2	Replicas: 2,3,1	Isr: 2,3,1	Elr: 	LastKnownElr: 
	Topic: demo-tp4	Partition: 1	Leader: 3	Replicas: 3,1,2	Isr: 3,1,2	Elr: 	LastKnownElr: 
	Topic: demo-tp4	Partition: 2	Leader: 1	Replicas: 1,2,3	Isr: 1,2,3	Elr: 	LastKnownElr: 


Une partition est **sous-répliquée** quand son ISR est plus courte que sa liste de répliques. L'outil Java ajoute la ligne `Elr` (Eligible Leader Replicas), que le client Python ne montre pas.

## 2. D'où vient une valeur ?

`describe_configs()` donne, pour chaque paramètre, sa valeur et son origine (`source`) : posée sur le topic, sur le cluster, ou valeur par défaut.

In [4]:
futur = list(admin.describe_configs([ConfigResource("topic", "demo-tp4")]).values())[0]
config = futur.result()
for nom in ("min.insync.replicas", "retention.ms", "unclean.leader.election.enable"):
    print("%-32s %-10s %s" % (nom, config[nom].value, ConfigSource(config[nom].source).name))

min.insync.replicas              2          DYNAMIC_TOPIC_CONFIG
retention.ms                     604800000  DEFAULT_CONFIG
unclean.leader.election.enable   false      STATIC_BROKER_CONFIG


## 3. Un producteur en arrière-plan

Le TP4 doit garder un producteur actif pendant qu'on arrête un nœud. Le producteur tourne donc dans un fil d'exécution : la cellule rend la main et on suit ses compteurs. Le TP fournit une classe `Charge` construite ainsi.

In [5]:
compteurs = {"confirmes": 0, "erreurs": 0}


def rapport(err, msg):
    compteurs["confirmes" if err is None else "erreurs"] += 1


def envoyer(nombre, debit):
    producteur = Producer({"bootstrap.servers": BS, "acks": "all", "log_level": 4})
    for i in range(nombre):
        producteur.produce("demo-tp4", value="M-%d" % i, on_delivery=rapport)
        producteur.poll(0)
        time.sleep(1 / debit)
    producteur.flush(30)


fil = threading.Thread(target=envoyer, args=(300, 100), daemon=True)
fil.start()
for _ in range(4):
    print(compteurs, "en cours" if fil.is_alive() else "termine")
    time.sleep(1)

{'confirmes': 0, 'erreurs': 0} en cours


{'confirmes': 95, 'erreurs': 0} en cours


{'confirmes': 190, 'erreurs': 0} en cours


{'confirmes': 285, 'erreurs': 0} en cours


## 4. Agir sur les conteneurs avec `docker compose`

Une cellule `%%bash` lance des commandes dans le dossier `lab/` : c'est ainsi que le TP arrête puis redémarre un nœud (`docker compose stop`, `docker compose start`). Ici, simple lecture de l'état des nœuds.

In [6]:
%%bash
cd $FORMATION_DIR/lab
docker compose ps --format '{{.Name}} {{.Status}}' kafka-1 kafka-2 kafka-3

kafka-1 Up 2 minutes (healthy)
kafka-2 Up 2 minutes (healthy)
kafka-3 Up 2 minutes (healthy)


## Ménage

In [7]:
fil.join(30)
print(compteurs)
admin.delete_topics(["demo-tp4"])["demo-tp4"].result()
print("demo-tp4 supprime")

{'confirmes': 300, 'erreurs': 0}
demo-tp4 supprime


## La méthode du TP4

Ouvrez `tps/jour1/tp4_fiabilite_panne/enonce.ipynb`.

- Remplissez les prédictions de l'étape 6.6 **avant** d'exécuter les productions : l'intérêt du TP est de confronter votre raisonnement au cluster.
- La cellule 6.4 lance la charge puis arrête `kafka-3` : exécutez-la une seule fois, d'un seul bloc.
- Pendant la panne, les commandes en ligne de commande sont plus lentes : `$BS` contient encore l'adresse du nœud arrêté.
- Ne redémarrez `kafka-3` qu'à l'étape 7.8.